In [ ]:
import subprocess
subprocess.run(["pip", "install", "transformers", "datasets", "seqeval",
                "accelerate", "-q"], check=True)
print("Packages ready.")

In [ ]:
import os, json

DATASET_PATH = "/kaggle/input/pii-masking-processed-dataset"

# Label config — must match Day 2 exactly
LABEL2ID = {"O": 0, "B-PER": 1, "I-PER": 2, "B-EMAIL": 3, "I-EMAIL": 4}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = 5

# Training runs: (model_name, seed, run_label)
RUNS = [
    ("microsoft/deberta-v3-small", 42, "deberta_seed42"),
    ("microsoft/deberta-v3-small",  0, "deberta_seed0"),
    ("microsoft/deberta-v3-small",  7, "deberta_seed7"),
    ("distilbert-base-cased",       42, "distilbert_seed42"),
]

# Shared hyperparameters — DO NOT change these
HP = {
    "learning_rate":                  3e-5,
    "per_device_train_batch_size":    16,
    "per_device_eval_batch_size":     32,
    "num_train_epochs":               5,
    "weight_decay":                   0.01,
    "warmup_ratio":                   0.1,
    "fp16":                           True,
    "dataloader_num_workers":         2,
    "save_total_limit":               1,
    "load_best_model_at_end":         True,
    "metric_for_best_model":          "eval_overall_f1",
    "greater_is_better":              True,
    "evaluation_strategy":            "epoch",
    "save_strategy":                  "epoch",
    "logging_steps":                  50,
    "report_to":                      "none",
}

EARLY_STOPPING_PATIENCE = 2
OUTPUT_DIR = "/kaggle/working"

print("Config loaded.")
print(f"Runs planned: {len(RUNS)}")

In [ ]:
from datasets import load_from_disk

dataset = load_from_disk(f"{DATASET_PATH}/hf_dataset")
print(dataset)
print("Columns:", dataset["train"].column_names)
print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))
print("Test size:", len(dataset["test"]))

In [ ]:
import numpy as np
from seqeval.metrics import f1_score

def compute_metrics_factory(id2label):
    def compute_metrics(p):
        predictions, labels = p
        predictions = np.argmax(predictions, axis=2)

        true_labels = [
            [id2label[l] for l in label if l != -100]
            for label in labels
        ]
        true_preds = [
            [id2label[pred] for pred, l in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]

        overall_f1  = f1_score(true_labels, true_preds)
        per_f1      = f1_score(true_labels, true_preds, average=None, labels=["PER"])
        email_f1    = f1_score(true_labels, true_preds, average=None, labels=["EMAIL"])

        # Token-level FPR and FNR
        tp = fp = fn = tn = 0
        for true_seq, pred_seq in zip(true_labels, true_preds):
            for t, p in zip(true_seq, pred_seq):
                is_true_entity = t != "O"
                is_pred_entity = p != "O"
                if is_true_entity and is_pred_entity:       tp += 1
                elif not is_true_entity and is_pred_entity: fp += 1
                elif is_true_entity and not is_pred_entity: fn += 1
                else:                                       tn += 1

        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

        return {
            "eval_overall_f1":    overall_f1,
            "eval_per_f1":        float(per_f1[0]) if len(per_f1) > 0 else 0.0,
            "eval_email_f1":      float(email_f1[0]) if len(email_f1) > 0 else 0.0,
            "eval_token_fpr":     fpr,
            "eval_token_fnr":     fnr,
        }
    return compute_metrics

In [ ]:
import torch
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification,
    EarlyStoppingCallback, set_seed
)

all_results = []

def train_one_run(model_name, seed, run_label):
    print(f"\n{'='*60}")
    print(f"START: {run_label}  |  model={model_name}  |  seed={seed}")
    print(f"{'='*60}\n")
    set_seed(seed)

    run_output_dir = f"{OUTPUT_DIR}/{run_label}"
    os.makedirs(run_output_dir, exist_ok=True)

    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # The dataset was already tokenized in Day 2; check if we need to re-tokenize
    sample = dataset["train"][0]
    if "input_ids" in sample:
        print("Using pre-tokenized dataset from Day 2.")
        tokenized_datasets = dataset
    else:
        print("Re-tokenizing dataset...")
        raise ValueError(
            "Dataset does not contain input_ids. Day 2 preprocessing may be incomplete. "
            "Check data/processed/hf_dataset/ locally."
        )

    # Model
    model = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )

    # Data collator
    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

    # Training arguments — build from shared HP dict
    training_args = TrainingArguments(
        output_dir=run_output_dir,
        seed=seed,
        **HP,
    )

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics_factory(ID2LABEL),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )

    # Train
    train_result = trainer.train()

    # Evaluate on validation
    val_metrics = trainer.evaluate(tokenized_datasets["validation"])
    print(f"\nValidation metrics: {val_metrics}")

    # Evaluate on test (only time test set is touched)
    test_metrics = trainer.evaluate(tokenized_datasets["test"])
    print(f"Test metrics: {test_metrics}")

    # Save model
    trainer.save_model(f"{run_output_dir}/best_model")

    # Collect result
    result = {
        "run_label":    run_label,
        "model_name":   model_name,
        "seed":         seed,
        "val_metrics":  {k: float(v) for k, v in val_metrics.items()},
        "test_metrics": {k: float(v) for k, v in test_metrics.items()},
        "train_loss":   float(train_result.training_loss),
        "train_steps":  train_result.global_step,
    }

    # Save per-run JSON immediately (so partial results survive if later run crashes)
    with open(f"{OUTPUT_DIR}/{run_label}_result.json", "w") as f:
        json.dump(result, f, indent=2)

    print(f"\nSaved: {run_label}_result.json")
    return result


# Execute all runs in order
for model_name, seed, run_label in RUNS:
    result = train_one_run(model_name, seed, run_label)
    all_results.append(result)

print("\n\n=== ALL RUNS COMPLETE ===")

In [ ]:
import pandas as pd

rows = []
for r in all_results:
    rows.append({
        "run":        r["run_label"],
        "model":      r["model_name"].split("/")[-1],
        "seed":       r["seed"],
        "val_f1":     r["val_metrics"].get("eval_overall_f1", 0),
        "test_f1":    r["test_metrics"].get("eval_overall_f1", 0),
        "per_f1":     r["test_metrics"].get("eval_per_f1", 0),
        "email_f1":   r["test_metrics"].get("eval_email_f1", 0),
        "token_fpr":  r["test_metrics"].get("eval_token_fpr", 0),
        "token_fnr":  r["test_metrics"].get("eval_token_fnr", 0),
        "train_loss": r["train_loss"],
    })

df = pd.DataFrame(rows)
print("\n=== RESULTS TABLE ===")
print(df.to_string(index=False))

deberta_df = df[df["model"] == "deberta-v3-small"]
print(f"\nDeBERTa-v3-small Test F1 (3-seed):")
print(f"  Mean: {deberta_df['test_f1'].mean():.4f}")
print(f"  Std:  {deberta_df['test_f1'].std():.4f}")

summary = {
    "runs": all_results,
    "deberta_test_f1_mean": float(deberta_df["test_f1"].mean()),
    "deberta_test_f1_std":  float(deberta_df["test_f1"].std()),
    "deberta_test_fpr_mean": float(deberta_df["token_fpr"].mean()),
    "deberta_test_fnr_mean": float(deberta_df["token_fnr"].mean()),
}
with open(f"{OUTPUT_DIR}/training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\ntraining_summary.json saved to /kaggle/working/")
print("Pull with: kaggle kernels output abdulmuizz28/pii-masking-day3-training -p results/")